In [1]:
# Load Doc

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

loader=PyPDFLoader("10 RAG Architectures.pdf")
documents=loader.load()

splitter= RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks=splitter.split_documents(documents)

#load Embedding model
embeddings=HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Embed all Chunks and store in FAISS
vectorstore=FAISS.from_documents(chunks,embeddings)

# Save to disk so you dont Re-embed every time
vectorstore.save_local("faiss_index")

# Load later
vectorstore= FAISS.load_local("faiss_index", embeddings, allow_dangerous_deserialization=True)


C:\Users\neha1\AppData\Local\Temp\ipykernel_20744\1449797950.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
c:\Users\neha1\Desktop\Skill Up\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8531.91it/s]


## Step 4 : Retrieve relevant chunks

When user asks a question-- embed the question, search FAISS for similar chunks return top Ke most relevant.

In [2]:
# Create retriever from vectorstore
retriver=vectorstore.as_retriever(
    search_kwargs={"k":3} # Return top three most similar chunks
)

# Retrieve for a query
query= "name some RAG architecture"
relevant_chunks= retriver.invoke(query)

# See what was retrieved
for chunk in relevant_chunks:
    print(chunk.page_content)
    print("---")

10  RAG  Architectures    1.  Standard  RAG  
The  baseline  Retrieval-Augmented  Generation  setup.  
A  user  query  is  embedded,  relevant  documents  are  retrieved  from  a  vector  
database,
 
and
 
the
 
LLM
 
generates
 
an
 
answer
 
grounded
 
in
 
that
 
context.
 
Example  
User:  “What  are  the  side  effects  of  ibuprofen?”  
 
System:
 
●  Retrieves  medical  documents  ●  Feeds  them  to  the  LLM  ●  Generates  a  grounded  answer  
Usage
---
7.  Attention-based  RAG  
Uses  attention  mechanisms  to  prioritize  the  most  relevant  parts  of  retrieved  
documents.
 
 
Instead
 
of
 
treating
 
all
 
retrieved
 
chunks
 
equally,
 
the
 
model:
 
●  assigns  weights  to  different  passages  ●  focuses  on  high-signal  content  during  generation  
 
Example  
User:  “Causes  of  climate  change?”  
 
System:
---
Optimizes  RAG  pipelines  under  cost  constraints  (tokens,  API  calls,  latency).  
 
The
 
system
 
dynamically:
 
●  limits  retrieval  depth  ● 

# What happens internally
1. Query → embedding model → query vector
2. FAISS searches all stored vectors for most similar
3. Returns top 3 chunks with their original text

# What does k=3 mean in the retriever?
 It means return the top 3 most similar chunks to the query. Higher k gives more context but also more noise. Lower k is more precise but might miss relevant information.